# Module D — FHIR Data Bridge: PUF vs. FHIR Comparison

**Track:** Interoperability Operations  
**Path:** `src/modules/fhir_data_bridge/`

This notebook maps MSSP PUF data elements to their FHIR R4 equivalents and surfaces the interoperability gaps in CMS-0057-F API coverage. It demonstrates what a FHIR-based query returns versus what CMS publishes in flat-file PUF format.

**Key finding target:** 4 MSSP PUF fields — including shared savings rate (`sav_rate`) and enrollment-type-stratified expenditure — have no direct FHIR R4 equivalent in any CMS-0057-F required API. CMS publishes these metrics only in flat-file format.

---
**References:**
- US Core Implementation Guide: https://hl7.org/fhir/us/core/  
- CMS Blue Button 2.0: https://bluebutton.cms.gov/developers/  
- CMS-0057-F required APIs: https://www.cms.gov/priorities/key-initiatives/burden-reduction/interoperability

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from puf_to_fhir_mapper import FHIR_FIELD_MAP, PUF_FIELD_CATALOG, map_puf_row_to_fhir
from mapping_report import build_fhir_mapping_table, build_api_coverage_summary

pd.set_option('display.max_colwidth', None)

## 1. Full PUF-to-FHIR mapping table

In [ ]:
# Build from the catalog (no PUF dataframe needed for the mapping table itself)
mapping_df = build_fhir_mapping_table(pd.DataFrame())

# Style: highlight gap rows
def highlight_gaps(row):
    color = "background-color: #ffe4e1; font-weight: bold" if row["gap"] else ""
    return [color] * len(row)

display_cols = ["puf_field", "fhir_path", "cms_0057f_api", "support_status", "gap", "note"]
mapping_df[display_cols].style.apply(highlight_gaps, axis=1)

## 2. API coverage bar chart — fields per CMS-0057-F API

In [ ]:
coverage_summary = build_api_coverage_summary(mapping_df)

fig = px.bar(
    coverage_summary,
    x="cms_0057f_api",
    y="field_count",
    color="support_status",
    barmode="stack",
    title="MSSP PUF Field Coverage by CMS-0057-F API",
    labels={
        "cms_0057f_api": "CMS API",
        "field_count":   "Number of PUF fields",
        "support_status":"FHIR Support Status",
    },
    color_discrete_map={
        "Yes":         "#2ca02c",
        "Partial":     "#ff7f0e",
        "No mapping":  "#d62728",
    },
)
fig.update_layout(xaxis_tickangle=-15)
fig.show()

gap_count = mapping_df["gap"].sum()
total = len(mapping_df)
print(f"Gap fields (no FHIR equivalent): {gap_count} of {total} mapped PUF fields ({gap_count/total:.0%})")

## 3. Gap field detail — what is missing and why

In [ ]:
gap_fields = mapping_df.loc[mapping_df["gap"] == True, ["puf_field", "support_status", "note"]]
print(f"\n{'='*80}")
print("INTEROPERABILITY GAP FIELDS")
print(f"{'='*80}")
for _, row in gap_fields.iterrows():
    print(f"\n  PUF field:  {row['puf_field']}")
    print(f"  Status:     {row['support_status']}")
    print(f"  Gap reason: {row['note']}")

## 4. Sample FHIR resource mapping from a PUF row

In [ ]:
import json

sample_puf_row = {
    "year":            "2024",
    "state_id":        "06",
    "county_id":       "037",
    "avg_risk_score":  1.24,
    "per_capita_exp":  13_450.00,
    "enrollment_type": "Aged Non-Dual",
    "n_ab_ben":        4_820,
    "quality_score":   0.87,
    # Gap fields — present in PUF but have no FHIR path:
    "sav_rate":                        0.023,
    "per_capita_exp_by_enrollment_type": 13_450.00,
}

fhir_output = map_puf_row_to_fhir(sample_puf_row)

print("FHIR R4 field mapping output (gap fields are excluded):")
print(json.dumps(fhir_output, indent=2, default=str))

unmapped = {k: v for k, v in sample_puf_row.items() if k not in FHIR_FIELD_MAP}
print(f"\nPUF fields with NO FHIR mapping: {list(unmapped.keys())}")

## 5. CMS Blue Button 2.0 sandbox — live query example

The cell below demonstrates a live ExplanationOfBenefit query against the CMS BB2 sandbox. Register for sandbox credentials at https://bluebutton.cms.gov/developers/ and replace `ACCESS_TOKEN` with a valid bearer token.

In [ ]:
# Uncomment and set your sandbox access token to run live queries.
# ACCESS_TOKEN = "your_sandbox_token_here"
#
# from bb2_sandbox_query import BlueButton2SandboxClient
# client = BlueButton2SandboxClient(access_token=ACCESS_TOKEN)
# eob_bundle = client.fetch_explanation_of_benefit(beneficiary_id="-20140000009893", count=5)
# entries = eob_bundle.get("entry", [])
# print(f"EOB entries returned: {len(entries)}")
# if entries:
#     print(json.dumps(entries[0]["resource"].get("total", []), indent=2))

print("BB2 sandbox query cell — set ACCESS_TOKEN and uncomment to run.")
print("Sandbox beneficiary IDs use negative integers, e.g. -20140000009893")

## 6. Key finding

> Four MSSP PUF fields — `sav_rate` (shared savings rate), `per_capita_exp_by_enrollment_type` (enrollment-stratified expenditure), and the per-capita/quality composite — have no direct FHIR R4 equivalent in any CMS-0057-F required API. CMS publishes these metrics exclusively in flat-file PUF format. This is a genuine interoperability gap: the FHIR resource model as implemented in US Core and the five CMS-0057-F APIs cannot express ACO financial performance or enrollment-type-stratified expenditure without custom extensions not currently defined in any required IG.

> This gap affects analytics vendors (Arcadia, Inovalon, Availity) that build FHIR-native data pipelines for ACO and MA analytics — they must maintain separate flat-file ingestion pathways for MSSP financial data alongside their FHIR API integrations.